In [2]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from cwfc.validation import rolling_origin_splits
from cwfc.metrics import mae
from cwfc.metrics import rmse
from cwfc.metrics import smape
import numpy as np
import pandas as pd

In [3]:
learning_dataframe = pd.read_csv("../learning_dataframe.csv")
new_dataframe = pd.DataFrame({"timestamp" : learning_dataframe["timestamp"], "aggregate_cpu_utilization" : learning_dataframe["aggregate_cpu_utilization"]})
data = new_dataframe["aggregate_cpu_utilization"]
median_values = []
median_values.append(9.256428)
for i in range(1, len(new_dataframe)):
    new_median = (data.iloc[max(0, i-4):i]).median()
    median_values.append(new_median)
new_dataframe["median_4"] = median_values  



In [4]:
print(new_dataframe)

     timestamp  aggregate_cpu_utilization  median_4
0            0                   9.396898  9.256428
1          300                   9.353775  9.396898
2          600                   9.454383  9.375336
3          900                   9.224419  9.396898
4         1200                   9.143503  9.375336
..         ...                        ...       ...
105      31500                   9.280858  9.244034
106      31800                   9.119565  9.244034
107      32100                   9.218538  9.184858
108      32400                   9.111634  9.234345
109      32700                   9.164556  9.169052

[110 rows x 3 columns]


In [5]:
splits = rolling_origin_splits(new_dataframe)
split = splits[0]
train = new_dataframe[new_dataframe["timestamp"] <= split.train_end].copy()
model = SGDRegressor(max_iter=1000, random_state=42)
scaler = StandardScaler()
train_array_scaled = scaler.fit_transform(train[["timestamp", "median_4"]])
model.fit(train_array_scaled, train["aggregate_cpu_utilization"])



new_df = pd.DataFrame({"timestamp": learning_dataframe["timestamp"], "aggregate_cpu_utilization": learning_dataframe["aggregate_cpu_utilization"]})
new_df["median_4"] = np.nan
validation = new_df[(new_df["timestamp"] >= split.validation_start) & (new_df["timestamp"] <= split.validation_end)].copy()
predictions_0 = []
history = train["aggregate_cpu_utilization"].tail(4).tolist()
for index in validation.index:
     median_value = np.median(history[-4:])
     new_df.loc[index, "median_4"] = median_value
     current_features = new_df.loc[[index], ["timestamp", "median_4"]]
     current_features_scaled = scaler.transform(current_features)
     prediction = model.predict(current_features_scaled)[0]
     predictions_0.append(prediction)
     history.append(prediction)
print(len(predictions_0))     
print(predictions_0)     

20
[np.float64(9.762789613805994), np.float64(9.744714859409923), np.float64(9.7312750041023), np.float64(9.696191308224067), np.float64(9.685917006083287), np.float64(9.669023525424954), np.float64(9.653361947500848), np.float64(9.644779251113222), np.float64(9.634100049180526), np.float64(9.626655061720633), np.float64(9.621149009506258), np.float64(9.616085706683522), np.float64(9.613035580372651), np.float64(9.61091230749369), np.float64(9.60974477968553), np.float64(9.60972139027453), np.float64(9.61043063055594), np.float64(9.612233065942673), np.float64(9.61422295012756), np.float64(9.617181172698238)]


In [6]:
MAE = mae(predictions_0 ,validation["aggregate_cpu_utilization"])
RMSE = rmse(predictions_0 ,validation["aggregate_cpu_utilization"])
SMAPE = smape(predictions_0 ,validation["aggregate_cpu_utilization"])
print(MAE)
print(RMSE)
print(SMAPE)


0.15121371812404094
0.2004876314701283
1.5856864570163516


In [7]:
split = splits[1]
train_1 = new_dataframe[new_dataframe["timestamp"] <= split.train_end].copy()
model_1 = SGDRegressor(max_iter=1000, random_state=42)
scaler_1 = StandardScaler()
train_array_scaled_1 = scaler_1.fit_transform(train_1[["timestamp", "median_4"]])
model_1.fit(train_array_scaled_1, train_1["aggregate_cpu_utilization"])



new_df_1 = pd.DataFrame({"timestamp": learning_dataframe["timestamp"], "aggregate_cpu_utilization": learning_dataframe["aggregate_cpu_utilization"]})
new_df_1["median_4"] = np.nan
validation_1 = new_df_1[(new_df_1["timestamp"] >= split.validation_start) & (new_df_1["timestamp"] <= split.validation_end)].copy()
predictions_1 = []
history_1 = train_1["aggregate_cpu_utilization"].tail(4).tolist()
for index in validation_1.index:
     median_value_1 = np.median(history_1[-4:])
     new_df_1.loc[index, "median_4"] = median_value_1
     current_features_1 = new_df_1.loc[[index], ["timestamp", "median_4"]]
     current_features_scaled_1 = scaler_1.transform(current_features_1)
     prediction_1 = model_1.predict(current_features_scaled_1)[0]
     predictions_1.append(prediction_1)
     history_1.append(prediction_1)
print(len(predictions_1))     
print(predictions_1)     

20
[np.float64(9.633904660982227), np.float64(9.634546977002353), np.float64(9.624954182945652), np.float64(9.61840463783042), np.float64(9.617751946270728), np.float64(9.615233959690972), np.float64(9.615336122007372), np.float64(9.616743567866228), np.float64(9.61880153260722), np.float64(9.621177907754507), np.float64(9.624648537590465), np.float64(9.628425121190432), np.float64(9.632647774703084), np.float64(9.637312564759245), np.float64(9.642214817422074), np.float64(9.64739753441985), np.float64(9.652794844908085), np.float64(9.658355698494006), np.float64(9.664072873915682), np.float64(9.66990945156189)]


In [8]:
MAE_1 = mae(predictions_1 ,validation_1["aggregate_cpu_utilization"])
RMSE_1 = rmse(predictions_1 ,validation_1["aggregate_cpu_utilization"])
SMAPE_1 = smape(predictions_1 ,validation_1["aggregate_cpu_utilization"])
print(MAE_1)
print(RMSE_1)
print(SMAPE_1)


0.26222539304547376
0.2936897456354252
2.7685623946991886


In [9]:
split = splits[2]
train_2 = new_dataframe[new_dataframe["timestamp"] <= split.train_end].copy()
model_2 = SGDRegressor(max_iter=1000, random_state=42)
scaler_2 = StandardScaler()
train_array_scaled_2 = scaler_2.fit_transform(train_2[["timestamp", "median_4"]])
model_2.fit(train_array_scaled_2, train_2["aggregate_cpu_utilization"])



new_df_2 = pd.DataFrame({"timestamp": learning_dataframe["timestamp"], "aggregate_cpu_utilization": learning_dataframe["aggregate_cpu_utilization"]})
new_df_2["median_4"] = np.nan
validation_2 = new_df_2[(new_df_2["timestamp"] >= split.validation_start) & (new_df_2["timestamp"] <= split.validation_end)].copy()
predictions_2 = []
history_2 = train_2["aggregate_cpu_utilization"].tail(4).tolist()
for index in validation_2.index:
     median_value_2 = np.median(history_2[-4:])
     new_df_2.loc[index, "median_4"] = median_value_2
     current_features_2 = new_df_2.loc[[index], ["timestamp", "median_4"]]
     current_features_scaled_2 = scaler_2.transform(current_features_2)
     prediction_2 = model_2.predict(current_features_scaled_2)[0]
     predictions_2.append(prediction_2)
     history_2.append(prediction_2)
print(len(predictions_2))     
print(predictions_2)     

20
[np.float64(9.289965059197346), np.float64(9.290855273656446), np.float64(9.291031853133575), np.float64(9.291704654808894), np.float64(9.291964895296484), np.float64(9.292553616932295), np.float64(9.293174692788233), np.float64(9.293763252372953), np.float64(9.294491357809113), np.float64(9.295219400575258), np.float64(9.295988834887828), np.float64(9.296812211527069), np.float64(9.297651571281474), np.float64(9.298527799501832), np.float64(9.29943106998626), np.float64(9.30035477976111), np.float64(9.30130320573071), np.float64(9.302269994240488), np.float64(9.303254245729725), np.float64(9.30425515705686)]


In [10]:
MAE_2 = mae(predictions_2 ,validation_2["aggregate_cpu_utilization"])
RMSE_2 = rmse(predictions_2 ,validation_2["aggregate_cpu_utilization"])
SMAPE_2 = smape(predictions_2 ,validation_2["aggregate_cpu_utilization"])
print(MAE_2)
print(RMSE_2)
print(SMAPE_2)


0.08008124059227954
0.094307133374682
0.8615992174996661


In [11]:
split = splits[3]
train_3 = new_dataframe[new_dataframe["timestamp"] <= split.train_end].copy()
model_3 = SGDRegressor(max_iter=1000, random_state=42)
scaler_3 = StandardScaler()
train_array_scaled_3 = scaler_3.fit_transform(train_3[["timestamp", "median_4"]])
model_3.fit(train_array_scaled_3, train_3["aggregate_cpu_utilization"])



new_df_3 = pd.DataFrame({"timestamp": learning_dataframe["timestamp"], "aggregate_cpu_utilization": learning_dataframe["aggregate_cpu_utilization"]})
new_df_3["median_4"] = np.nan
validation_3 = new_df_3[(new_df_3["timestamp"] >= split.validation_start) & (new_df_3["timestamp"] <= split.validation_end)].copy()
predictions_3 = []
history_3 = train_3["aggregate_cpu_utilization"].tail(4).tolist()
for index in validation_3.index:
     median_value_3 = np.median(history_3[-4:])
     new_df_3.loc[index, "median_4"] = median_value_3
     current_features_3 = new_df_3.loc[[index], ["timestamp", "median_4"]]
     current_features_scaled_3 = scaler_3.transform(current_features_3)
     prediction_3 = model_3.predict(current_features_scaled_3)[0]
     predictions_3.append(prediction_3)
     history_3.append(prediction_3)
print(len(predictions_3))     
print(predictions_3)     

20
[np.float64(9.270173855733503), np.float64(9.245139762369817), np.float64(9.237361807397063), np.float64(9.237155660698452), np.float64(9.223486235300205), np.float64(9.220004091173282), np.float64(9.21410458780454), np.float64(9.206860893242844), np.float64(9.202805314969563), np.float64(9.197206315775807), np.float64(9.192363906881045), np.float64(9.188196341320888), np.float64(9.183705927280263), np.float64(9.179802852065658), np.float64(9.176044205652738), np.float64(9.172394083620832), np.float64(9.169044216891923), np.float64(9.165798140671965), np.float64(9.162719792926506), np.float64(9.159807231276382)]


In [12]:
MAE_3 = mae(predictions_3 ,validation_3["aggregate_cpu_utilization"])
RMSE_3 = rmse(predictions_3 ,validation_3["aggregate_cpu_utilization"])
SMAPE_3 = smape(predictions_3 ,validation_3["aggregate_cpu_utilization"])
print(MAE_3)
print(RMSE_3)
print(SMAPE_3)


0.08469626465226643
0.10345456836465104
0.9155080048362512


In [13]:
test_dataframe = pd.read_csv("../test_dataframe.csv")

train_final = new_dataframe.copy()
model_final = SGDRegressor(max_iter = 1000, random_state = 42)
scaler_final = StandardScaler()
train_array_scaled_final= scaler_final.fit_transform(train_final[["timestamp", "median_4"]])
model_final.fit(train_array_scaled_final, train_final["aggregate_cpu_utilization"])


new_df_final = pd.DataFrame({"timestamp": test_dataframe["timestamp"], "aggregate_cpu_utilization": test_dataframe["aggregate_cpu_utilization"]})
new_df_final["median_4"] = np.nan
validation_final = new_df_final.copy()
predictions_final = []
history_final = train_final["aggregate_cpu_utilization"].tail(4).tolist()
for index in validation_final.index:
     median_value_final = np.median(history_final[-4:])
     new_df_final.loc[index, "median_4"] = median_value_final
     current_features_final = new_df_final.loc[[index], ["timestamp", "median_4"]]
     current_features_scaled_final = scaler_final.transform(current_features_final)
     prediction_final = model_final.predict(current_features_scaled_final)[0]
     predictions_final.append(prediction_final)
     history_final.append(prediction_final)
print(len(predictions_final))     
print(predictions_final)  


24
[np.float64(9.124735374245024), np.float64(9.12640839314257), np.float64(9.109999964999645), np.float64(9.109514933202522), np.float64(9.102180803184966), np.float64(9.095342554934058), np.float64(9.091593700008742), np.float64(9.085192927231901), np.float64(9.080288696417531), np.float64(9.075567073739954), np.float64(9.070363181653223), np.float64(9.06586018936363), np.float64(9.061232114435441), np.float64(9.056695298766718), np.float64(9.052398836911383), np.float64(9.048088256723458), np.float64(9.043916096284661), np.float64(9.039838369573182), np.float64(9.035812527941944), np.float64(9.031883882478262), np.float64(9.028016312449456), np.float64(9.024210970935172), np.float64(9.020471694112835), np.float64(9.016783886014561)]


In [14]:
predictions = pd.DataFrame({"timestamp" : test_dataframe["timestamp"], "active_vm_count" : 10000, "aggregate_cpu_utilization" : predictions_final})
print(predictions)

    timestamp  active_vm_count  aggregate_cpu_utilization
0       33000            10000                   9.124735
1       33300            10000                   9.126408
2       33600            10000                   9.110000
3       33900            10000                   9.109515
4       34200            10000                   9.102181
5       34500            10000                   9.095343
6       34800            10000                   9.091594
7       35100            10000                   9.085193
8       35400            10000                   9.080289
9       35700            10000                   9.075567
10      36000            10000                   9.070363
11      36300            10000                   9.065860
12      36600            10000                   9.061232
13      36900            10000                   9.056695
14      37200            10000                   9.052399
15      37500            10000                   9.048088
16      37800 

In [15]:
predictions.to_csv("../predictions/predictions_notebook.csv", index=False)
train_final.to_csv("../train_final.csv", index=False)

In [16]:
MAE_final= mae(predictions_final ,validation_final["aggregate_cpu_utilization"])
RMSE_final = rmse(predictions_final ,validation_final["aggregate_cpu_utilization"])
SMAPE_final = smape(predictions_final ,validation_final["aggregate_cpu_utilization"])
print(MAE_final)
print(RMSE_final)
print(SMAPE_final)


0.2260727454279777
0.2704134752248967
2.455807896730089


In [17]:
np.save("../predictions/sgd_predictions.npy", predictions_final)

